# Data Splitting ABSA Hotel Santika - Kaggle Ready

Notebook ini digunakan untuk membagi dataset final audit `dataset_absa_labeled_v3_final_audited.csv` menjadi data `train`, `validation`, dan `test` untuk fine-tuning model IndoBERT ABSA.

Rasio split yang digunakan adalah **80% train, 10% validation, 10% test**.

Alasan pemilihan rasio:

- Dataset memiliki 14.747 review, sehingga 10% validation dan 10% test masih menghasilkan sekitar 1.475 review per subset. Jumlah ini cukup untuk evaluasi hold-out, sambil tetap mempertahankan mayoritas data untuk training.
- Model yang digunakan adalah model berbasis BERT/IndoBERT. Paper BERT menunjukkan pola fine-tuning supervised pada downstream task dengan pemisahan data training dan development/evaluation set, sehingga validation set tetap diperlukan untuk memilih model/checkpoint tanpa menyentuh test set.
- Karena dataset ABSA bersifat multi-aspect dan multi-label, split tidak boleh random biasa. Paper Sechidis, Tsoumakas, dan Vlahavas (2011) membahas pentingnya stratifikasi pada multi-label data agar proporsi label tetap terjaga di setiap subset.
- Kohavi (1995) menekankan pentingnya estimasi performa dan model selection yang reliabel. Dalam konteks ini, validation dipakai untuk model selection, sedangkan test disimpan sebagai evaluasi akhir.

Referensi utama:

- Devlin et al. (2019), *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*: https://arxiv.org/abs/1810.04805
- Sechidis et al. (2011), *On the Stratification of Multi-label Data*: https://doi.org/10.1007/978-3-642-23808-6_10
- Kohavi (1995), *A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection*: https://www.ijcai.org/Proceedings/95-2/Papers/016.pdf


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ASPECTS = ['Kenyamanan', 'Kebersihan', 'Pelayanan', 'Harga', 'Lokasi', 'Fasilitas', 'Makanan']
LABELS = ['positif', 'negatif', 'netral', 'none']
SPLIT_RATIOS = {'train': 0.80, 'validation': 0.10, 'test': 0.10}

OUTPUT_DIR = Path('/kaggle/working/absa_santika_split_v3') if Path('/kaggle').exists() else Path('absa_santika_split_v3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Output directory:', OUTPUT_DIR.resolve())

## 1. Load Dataset

Di Kaggle, upload dataset sebagai input dataset. Notebook ini akan mencari file `dataset_absa_labeled_v3_final_audited.csv` secara otomatis di `/kaggle/input`.

In [ ]:
def find_dataset_path():
    candidate_names = [
        'dataset_absa_labeled_v3_final_audited.csv',
        'dataset_absa_labeled_v2_audited.csv',
        'dataset_absa_labeled.csv',
    ]
    search_roots = []
    if Path('/kaggle/input').exists():
        search_roots.append(Path('/kaggle/input'))
    search_roots.append(Path.cwd())
    search_roots.append(Path('/kaggle/working'))

    for root in search_roots:
        if not root.exists():
            continue
        for name in candidate_names:
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        'Dataset tidak ditemukan. Upload dataset ke Kaggle input dengan nama '
        'dataset_absa_labeled_v3_final_audited.csv atau letakkan di working directory.'
    )

DATASET_PATH = find_dataset_path()
print('Dataset path:', DATASET_PATH)

df = pd.read_csv(DATASET_PATH)
print('Shape:', df.shape)
df.head(3)

## 2. Validasi Struktur Dataset

Tahap ini memastikan kolom aspek dan kolom alasan tersedia, label dinormalisasi, serta tidak ada duplikasi `ID_Review`.

In [ ]:
REQUIRED_COLUMNS = [
    'ID_Review', 'Platform', 'Nama_Hotel', 'Review_Date', 'Text_Review',
    *ASPECTS,
    *[f'Alasan_{aspect}' for aspect in ASPECTS],
]

missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f'Kolom wajib tidak ditemukan: {missing_columns}')

if df['ID_Review'].duplicated().any():
    duplicated = df.loc[df['ID_Review'].duplicated(), 'ID_Review'].head().tolist()
    raise ValueError(f'Terdapat ID_Review duplikat, contoh: {duplicated}')

df = df.copy()
for aspect in ASPECTS:
    df[aspect] = (
        df[aspect]
        .fillna('none')
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'nan': 'none', '': 'none', '-': 'none'})
    )
    invalid = sorted(set(df[aspect].unique()) - set(LABELS))
    if invalid:
        raise ValueError(f'Label tidak valid pada aspek {aspect}: {invalid}')

print('Validasi struktur dataset berhasil.')
print(df[ASPECTS].apply(lambda col: col.value_counts()).fillna(0).astype(int))

## 3. Membuat Matriks Stratifikasi Multi-Aspect

ABSA Hotel Santika bukan klasifikasi single-label biasa. Satu review dapat memiliki banyak aspek sekaligus, dan setiap aspek memiliki label sentimen masing-masing. Karena itu, notebook ini membuat matriks biner `aspect__label` untuk menjaga distribusi label per aspek.

In [ ]:
def build_stratification_matrix(dataframe, include_metadata=True):
    columns = []
    matrix_parts = []

    for aspect in ASPECTS:
        for label in LABELS:
            col_name = f'{aspect}__{label}'
            columns.append(col_name)
            matrix_parts.append((dataframe[aspect] == label).astype(np.int8).to_numpy())

    if include_metadata:
        # Metadata tidak menjadi target model, tetapi dibantu agar distribusi platform/hotel tidak terlalu timpang.
        for meta_col in ['Platform', 'Nama_Hotel']:
            if meta_col in dataframe.columns:
                counts = dataframe[meta_col].fillna('unknown').astype(str).value_counts()
                kept_values = counts[counts >= 20].index.tolist()
                for value in kept_values:
                    col_name = f'{meta_col}__{value}'
                    columns.append(col_name)
                    matrix_parts.append((dataframe[meta_col].fillna('unknown').astype(str) == value).astype(np.int8).to_numpy())

    y = np.vstack(matrix_parts).T
    return y, columns

Y, stratification_columns = build_stratification_matrix(df, include_metadata=True)
print('Stratification matrix:', Y.shape)
print('Jumlah indikator stratifikasi:', len(stratification_columns))
print(stratification_columns[:10])

## 4. Iterative Multi-Label Stratified Split

Kaggle tidak selalu menyediakan library khusus seperti `scikit-multilearn`. Karena itu, notebook ini memakai implementasi greedy iterative stratification sederhana agar tetap bisa running tanpa instalasi tambahan.

In [ ]:
def compute_target_sizes(n_rows, ratios):
    split_names = list(ratios.keys())
    raw_sizes = np.array([ratios[name] * n_rows for name in split_names])
    sizes = np.floor(raw_sizes).astype(int)
    remainder = n_rows - sizes.sum()
    fractional_order = np.argsort(-(raw_sizes - sizes))
    for idx in fractional_order[:remainder]:
        sizes[idx] += 1
    return dict(zip(split_names, sizes))


def iterative_multilabel_split(y, ratios, seed=42):
    rng = np.random.default_rng(seed)
    n_rows, n_labels = y.shape
    split_names = list(ratios.keys())
    target_sizes = compute_target_sizes(n_rows, ratios)
    target_label_counts = {
        name: y.sum(axis=0) * ratios[name]
        for name in split_names
    }
    current_sizes = {name: 0 for name in split_names}
    current_label_counts = {name: np.zeros(n_labels, dtype=float) for name in split_names}

    label_frequency = y.sum(axis=0).astype(float)
    label_frequency[label_frequency == 0] = 1.0
    rarity = (y / label_frequency).sum(axis=1)
    cardinality = y.sum(axis=1)
    noise = rng.random(n_rows) * 1e-6
    order = np.lexsort((-noise, -cardinality, -rarity))

    assignment = np.empty(n_rows, dtype=object)

    for row_idx in order:
        active = y[row_idx] == 1
        best_split = None
        best_score = -np.inf

        for split_name in split_names:
            if current_sizes[split_name] >= target_sizes[split_name]:
                continue

            label_need = np.maximum(target_label_counts[split_name] - current_label_counts[split_name], 0)
            label_target = np.maximum(target_label_counts[split_name], 1e-9)
            relative_label_need = label_need / label_target
            label_need_score = relative_label_need[active].sum() if active.any() else 0.0
            size_need = target_sizes[split_name] - current_sizes[split_name]
            size_need_score = size_need / max(target_sizes[split_name], 1)

            score = label_need_score + size_need_score
            if score > best_score:
                best_score = score
                best_split = split_name

        if best_split is None:
            # Fallback jika semua kapasitas penuh karena pembulatan.
            best_split = min(split_names, key=lambda name: current_sizes[name] / max(target_sizes[name], 1))

        assignment[row_idx] = best_split
        current_sizes[best_split] += 1
        current_label_counts[best_split] += y[row_idx]

    return assignment, target_sizes


split_assignment, target_sizes = iterative_multilabel_split(Y, SPLIT_RATIOS, seed=SEED)
df['split'] = split_assignment

print('Target sizes:', target_sizes)
print('Actual sizes:')
print(df['split'].value_counts().reindex(SPLIT_RATIOS.keys()))

## 5. Evaluasi Distribusi Split

Bagian ini mengecek apakah distribusi label per aspek relatif terjaga antara full dataset, train, validation, dan test.

In [ ]:
def make_aspect_distribution_summary(dataframe):
    rows = []
    full_n = len(dataframe)
    for aspect in ASPECTS:
        full_counts = dataframe[aspect].value_counts().to_dict()
        for label in LABELS:
            full_count = int(full_counts.get(label, 0))
            full_pct = full_count / full_n if full_n else 0
            for split_name in SPLIT_RATIOS.keys():
                subset = dataframe[dataframe['split'] == split_name]
                split_count = int((subset[aspect] == label).sum())
                split_pct = split_count / len(subset) if len(subset) else 0
                rows.append({
                    'aspect': aspect,
                    'label': label,
                    'split': split_name,
                    'full_count': full_count,
                    'full_pct': full_pct,
                    'split_count': split_count,
                    'split_pct': split_pct,
                    'abs_pct_diff': abs(split_pct - full_pct),
                })
    return pd.DataFrame(rows)


aspect_summary = make_aspect_distribution_summary(df)
max_diff = aspect_summary['abs_pct_diff'].max()
mean_diff = aspect_summary['abs_pct_diff'].mean()

print(f'Max absolute percentage difference: {max_diff:.4f}')
print(f'Mean absolute percentage difference: {mean_diff:.4f}')
display(aspect_summary.sort_values('abs_pct_diff', ascending=False).head(20))

In [ ]:
def make_metadata_summary(dataframe, column):
    rows = []
    if column not in dataframe.columns:
        return pd.DataFrame(rows)
    full_counts = dataframe[column].fillna('unknown').astype(str).value_counts()
    for value, full_count in full_counts.items():
        if full_count < 20:
            continue
        full_pct = full_count / len(dataframe)
        for split_name in SPLIT_RATIOS.keys():
            subset = dataframe[dataframe['split'] == split_name]
            split_count = int((subset[column].fillna('unknown').astype(str) == value).sum())
            split_pct = split_count / len(subset) if len(subset) else 0
            rows.append({
                'column': column,
                'value': value,
                'split': split_name,
                'full_count': int(full_count),
                'full_pct': full_pct,
                'split_count': split_count,
                'split_pct': split_pct,
                'abs_pct_diff': abs(split_pct - full_pct),
            })
    return pd.DataFrame(rows)

metadata_summary = pd.concat([
    make_metadata_summary(df, 'Platform'),
    make_metadata_summary(df, 'Nama_Hotel'),
], ignore_index=True)

display(metadata_summary.sort_values('abs_pct_diff', ascending=False).head(20))

## 6. Simpan Output Split

Output akan tersimpan di `/kaggle/working/absa_santika_split_v3` sehingga bisa langsung di-download dari Kaggle.

In [ ]:
train_df = df[df['split'] == 'train'].drop(columns=['split']).copy()
validation_df = df[df['split'] == 'validation'].drop(columns=['split']).copy()
test_df = df[df['split'] == 'test'].drop(columns=['split']).copy()

# Sanity checks: semua baris harus terpakai tepat satu kali.
all_ids = set(df['ID_Review'])
train_ids = set(train_df['ID_Review'])
validation_ids = set(validation_df['ID_Review'])
test_ids = set(test_df['ID_Review'])

assert len(train_ids & validation_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(validation_ids & test_ids) == 0
assert train_ids | validation_ids | test_ids == all_ids

train_path = OUTPUT_DIR / 'train.csv'
validation_path = OUTPUT_DIR / 'validation.csv'
test_path = OUTPUT_DIR / 'test.csv'
full_with_split_path = OUTPUT_DIR / 'dataset_with_split.csv'
aspect_summary_path = OUTPUT_DIR / 'split_summary_by_aspect.csv'
metadata_summary_path = OUTPUT_DIR / 'split_summary_by_metadata.csv'
manifest_path = OUTPUT_DIR / 'split_manifest.json'

train_df.to_csv(train_path, index=False, encoding='utf-8-sig')
validation_df.to_csv(validation_path, index=False, encoding='utf-8-sig')
test_df.to_csv(test_path, index=False, encoding='utf-8-sig')
df.to_csv(full_with_split_path, index=False, encoding='utf-8-sig')
aspect_summary.to_csv(aspect_summary_path, index=False, encoding='utf-8-sig')
metadata_summary.to_csv(metadata_summary_path, index=False, encoding='utf-8-sig')

manifest = {
    'dataset_path': str(DATASET_PATH),
    'seed': SEED,
    'split_ratios': SPLIT_RATIOS,
    'target_sizes': {key: int(value) for key, value in target_sizes.items()},
    'actual_sizes': {key: int(value) for key, value in df['split'].value_counts().to_dict().items()},
    'n_rows': int(len(df)),
    'aspects': ASPECTS,
    'labels': LABELS,
    'max_abs_pct_diff_aspect_label': float(max_diff),
    'mean_abs_pct_diff_aspect_label': float(mean_diff),
    'method': 'greedy iterative multi-label stratified split using aspect-label indicators',
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

print('Saved files:')
for path in [train_path, validation_path, test_path, full_with_split_path, aspect_summary_path, metadata_summary_path, manifest_path]:
    print('-', path)

In [ ]:
print('Final split sizes')
print('Train:', train_df.shape)
print('Validation:', validation_df.shape)
print('Test:', test_df.shape)

display(pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(validation_df), len(test_df)],
    'percentage': [len(train_df)/len(df), len(validation_df)/len(df), len(test_df)/len(df)],
}))

display(aspect_summary.pivot_table(
    index=['aspect', 'label'],
    columns='split',
    values='split_count',
    aggfunc='sum'
).reset_index())

## Catatan untuk Fine-Tuning Berikutnya

- Gunakan `train.csv` untuk training.
- Gunakan `validation.csv` untuk memilih checkpoint terbaik, early stopping, dan tuning hyperparameter.
- Gunakan `test.csv` hanya sekali untuk evaluasi akhir setelah model final ditentukan.
- Jangan melakukan tuning berdasarkan test set agar evaluasi akhir tetap objektif.